# Overcooked-v2 environment scaffold — building from scratch

**Goal of this notebook** — build a *brand new* Overcooked-v2 environment (layout + recipes +
reward shaping) *without touching the RNN training code*. The training script
`baselines/IPPO/ippo_rnn_overcooked_v2.py` is deliberately structured so that everything you define here
reaches the trainer through exactly two hooks:

1. A **layout dict** (schema below) registered in
   `jaxmarl.environments.overcooked_v2.overcooked_v2_layouts` under a name of your choice.
2. **`ENV_KWARGS`** in `baselines/IPPO/config/ippo_rnn_overcooked_v2.yaml` — `layout`, `recipes`, `max_steps`,
   `random_reset`. The trainer looks the name up here:
   ```python
   layout_name = config['ENV_KWARGS']['layout']
   config['ENV_KWARGS']['layout'] = overcooked_v2_layouts[layout_name]
   env = jaxmarl.make(config['ENV_NAME'], **config['ENV_KWARGS'])
   env = LogWrapper(env, replace_info=False)
   ```
   (see `baselines/IPPO/ippo_rnn_overcooked_v2.py:1059-1060` and `:502,512`).

**Contract with the trainer** — the network is initialised from `env.observation_space().shape` and
`env.action_space(env.agents[0]).n + 1` (`ippo_rnn_overcooked_v2.py:548,553`). If your custom env changes
these shapes, you break weight loading and any pre-trained checkpoint referenced in the config.
Keep the obs shape `(W, H, 28)` and the 6 discrete actions untouched — modify only the grid contents,
recipes, and reward shaping.

**How the notebook is structured** — each concept is one explanation cell + one demo cell
showing the existing code (heavily commented) + one YOUR TURN cell that tells you exactly what
to write and what the downstream implications are.

**Building from scratch, not remixing** — since you're authoring a new env end-to-end,
the first thing to nail down is what `overcooked_v2_layouts` *is* and what a layout dict
*must* look like for `Overcooked_v2.__init__` to accept it. That's section 2 below. After
that, section 3 gives you two authoring paths: the ASCII-grid shortcut (nice for quick
sketches) and the direct-dict path (what you'll want if your layout doesn't fit the parser's
assumptions or if you want to programmatically generate layouts).


## 1. Imports

These are the pieces the training script pulls in. We only need the environment-side subset here
(no `flax`, no `optax`, no `ActorCriticRNN`). If you can run this cell without errors, your conda
env `emergent_partner_model` is wired up correctly.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from flax.core.frozen_dict import FrozenDict

# Env factory + registered layout catalogue.
import jaxmarl
from jaxmarl.environments.overcooked_v2 import overcooked_v2_layouts
from jaxmarl.environments.overcooked_v2.layouts import layout_grid_to_dict

# LogWrapper is what the trainer wraps the env in before use.
from jaxmarl.wrappers.baselines import LogWrapper

# Optional: only needed if you want to render a rollout inline.
from jaxmarl.viz.overcooked_visualizer_v2 import OvercookedVisualizer

print('jax:', jax.__version__, '  devices:', jax.devices())


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Add any extra imports you personally want (e.g. matplotlib for a heatmap of your layout).
# 
# IMPLICATIONS: this cell is a scratchpad. Nothing here is consumed by the trainer — you can
# add or remove imports freely without breaking anything downstream.
# ------------------------------------------------------------------------------



## 2. What is `overcooked_v2_layouts`?

This is the single object the trainer looks your env up in, so understanding it precisely
matters more than anything else in the notebook.

**Type.** A plain Python `dict` (mutable at runtime), built in
`jaxmarl/environments/overcooked_v2/layouts.py:660` and then extended with auto-generated
variants (`layouts.py:701-709`) at import time.

**Contents.** ~2626 entries total = **26 hand-authored base layouts** + **2600 auto-generated variants**
(100 per base). Base names include:
`cramped_room`, `cramped_room_v2`, `cramped_room_v3`, `cramped_room_v4`, `fivebyfive_v1`,
`bottleneck_room`, `asymm_advantages`, `coord_ring`, `coord_ring_v2`, `coord_ring2_tomato`,
`forced_coord`, `forced_coord2`, `semi_forced_coord_v1`, `semi_forced_coord_v2`,
`counter_circuit`, `counter_circuit_onion`, `counter_circuit_onion2`,
`cramped_room_tomatoes`, `asymm_advantages_tomatoes`, `new_layout`, `big_room`,
`partial_divider`, `two_dividers`, `islands_in_middle`, `unusual_shape`, `cone_shape`.
Variants are named `<base>_variant_<i>_pots<n>_onions<n>_tomatoes<n>` and are produced by
`generate_variants(...)` which re-samples station positions while keeping the walls fixed.

**Value schema — this is your target.** Each value is a `FrozenDict` (from `flax.core.frozen_dict`)
with exactly these keys and types (from inspecting `overcooked_v2_layouts['cramped_room_v3']`):

| key                | type              | shape/dtype       | meaning |
|--------------------|-------------------|-------------------|---------|
| `height`           | `int`             | scalar            | grid rows |
| `width`            | `int`             | scalar            | grid cols |
| `wall_idx`         | `jnp.ndarray`     | `(K,)  int32`     | flat indices of all wall + station cells |
| `agent_idx`        | `jnp.ndarray`     | `(2,)  int32`     | flat indices of the two agent spawn cells |
| `goal_idx`         | `jnp.ndarray`     | `(G,)  int32`     | serving counter positions |
| `plate_pile_idx`   | `jnp.ndarray`     | `(P,)  int32`     | plate/bowl pile positions |
| `onion_pile_idx`   | `jnp.ndarray`     | `(O,)  int32`     | onion pile positions |
| `tomato_pile_idx`  | `jnp.ndarray`     | `(T,)  int32` or `(0,) float32` if unused | tomato pile positions |
| `pot_idx`          | `jnp.ndarray`     | `(Q,)  int32`     | pot positions |

**Position convention.** Every index is a flat, row-major cell id: `idx = row * width + col`,
where `(0,0)` is top-left. The constructor at `overcooked.py:256` reads `layout['height']`,
`layout['width']`, then uses each `*_idx` array to place objects onto a `(height, width)` grid.

**Wall bookkeeping quirk.** Every station cell (`goal_idx`, `plate_pile_idx`, `onion_pile_idx`,
`tomato_pile_idx`, `pot_idx`) is *also* listed in `wall_idx`. Agents cannot stand on stations —
they interact from an adjacent floor tile — so from the pathing code's point of view stations
are walls. If you build the dict yourself, remember to include station cells in `wall_idx` too;
the `layout_grid_to_dict` parser does this automatically (see `layouts.py:370-372`).

**Empty station type.** `tomato_pile_idx: jnp.array([])` is the idiom for 'no tomato piles
on this map'. The constructor tolerates the dtype being `float32` in that case (that's what
`jnp.array([])` returns by default) — the array is only accessed if `len > 0`.

**Mutability.** The dict itself is mutable, but the *values* are `FrozenDict`s (hashable,
cheap to pass through JAX). Overwriting a key is fine; mutating a value in-place is not.


In [ ]:
# EXISTING inspection — verify what you're targeting.
print('type of catalogue :', type(overcooked_v2_layouts).__name__)
print('total entries     :', len(overcooked_v2_layouts))
base = [k for k in overcooked_v2_layouts if '_variant_' not in k]
variants = [k for k in overcooked_v2_layouts if '_variant_' in k]
print('base layouts      :', len(base))
print('generated variants:', len(variants))
print()
print('schema of one entry (cramped_room_v3):')
d = overcooked_v2_layouts['cramped_room_v3']
print('  value type      :', type(d).__name__)
for k, v in d.items():
    kind = type(v).__name__
    extra = f' shape={v.shape} dtype={v.dtype}' if hasattr(v, 'shape') else ''
    print(f'  {k:>18}: {kind}{extra}')


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Sanity-inspect two more entries so you internalise the schema variance across the catalogue:
# 
#     for name in ['cramped_room', 'asymm_advantages', 'counter_circuit_onion']:
#         d = overcooked_v2_layouts[name]
#         print(name, '->', {k: (v.shape if hasattr(v, 'shape') else v) for k, v in d.items()})
# 
# IMPLICATIONS: your from-scratch layout dict must have **exactly** the 9 keys listed in the
# schema table above and station cells duplicated into `wall_idx`. Missing keys crash the
# constructor at `overcooked.py:498-540` (each is fetched with `layout.get(...)`, which returns
# `None`, which then blows up on array ops). Extra keys are silently ignored, but there's no
# reason to add any.
# ------------------------------------------------------------------------------



## 3. Authoring path A — describe your kitchen as an ASCII grid

An Overcooked-v2 layout is authored as a multi-line string. `layout_grid_to_dict` parses it
into flat position indices (`idx = row * width + col`, row-major). The symbol table
(from `jaxmarl/environments/overcooked_v2/layouts.py:330`):

| symbol | meaning                        | also counts as wall? |
|--------|--------------------------------|----------------------|
| `W`    | wall                           | yes                  |
| `A`    | agent spawn (need exactly 2)   | no                   |
| `X`    | goal / serving counter         | yes                  |
| `B`    | plate (bowl) pile              | yes                  |
| `O`    | onion pile                     | yes                  |
| `T`    | tomato pile                    | yes                  |
| `P`    | pot                            | yes                  |
| ` `    | empty floor tile               | no                   |

Anything that is a station (`X B O T P`) is *also* implicitly a wall — agents can only face and
`interact` with these from an adjacent floor tile, they can never stand on them.


In [ ]:
# EXISTING code — this is the actual `cramped_room_v3` layout that the current
# training config points at (`baselines/IPPO/config/ippo_rnn_overcooked_v2.yaml`).
cramped_room_v3 = """
WWWWW
P   O
W   W
BA AW
WWXWW
"""

# The two `A`s are the two agents. The pot `P` is top-left, onion pile `O` is top-right,
# plate pile `B` is bottom-left, serving goal `X` is bottom-centre. Everything else is wall
# or floor. Because ingredients (only onions here) and pot and plate and goal are all
# reachable, the task is solvable.
print(cramped_room_v3)


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Design your own kitchen as a string and bind it to `my_layout_grid`.
# 
# HARD CONSTRAINTS (violating these means the env constructor will crash or the task
# will silently be unsolvable):
#   - Exactly two `A` characters (the env hard-codes `num_agents=2`).
#   - Every row must be the same width (the parser reads width from row 0).
#   - The perimeter should be walls, or agents can walk off the map.
#   - Include at least one `P` (pot), one `O` and/or `T` (matching your recipes below),
#     one `B` (plate pile) and one `X` (goal). Otherwise no reward is ever emitted.
#   - Each station must be adjacent to at least one floor tile, otherwise agents can never
#     interact with it.
# 
# IMPLICATIONS: this raw string never touches the env directly — it has to be parsed into the
# position-index dict (next cell). Changing the width/height here does NOT change the network
# shape, because observations are always `(W, H, 28)`; the network's spatial dims flex with
# the env you pass in at init time. It DOES invalidate any pre-trained checkpoint that was
# trained on a different-sized grid, though — checkpoints from `cramped_room_v3` (5x5) won't
# load into e.g. a 7x7 env.
# ------------------------------------------------------------------------------



## 4. Path A continued — parsing the grid string into the layout dict

`Overcooked_v2.__init__` expects a `dict`-like layout with these keys
(`jaxmarl/environments/overcooked_v2/overcooked.py:256`):
`height`, `width`, `wall_idx`, `agent_idx`, `goal_idx`, `plate_pile_idx`,
`onion_pile_idx`, `tomato_pile_idx`, `pot_idx`. All `*_idx` fields are 1-D `jnp.ndarray` of flat
positions (`row*width + col`). `layout_grid_to_dict` does the conversion for you.


In [ ]:
# EXISTING helper. Two things worth noticing in its output:
#   (a) station symbols (X B O T P) end up in BOTH their own list AND `wall_idx`.
#   (b) agent positions do NOT end up in `wall_idx` — agents stand on floor.
example_layout = layout_grid_to_dict(cramped_room_v3)
for k, v in example_layout.items():
    print(f'{k:>18}: {v}')
print()
print('num agents:', len(example_layout['agent_idx']))
print('grid size :', example_layout['height'], 'x', example_layout['width'])


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Convert your grid to a dict:
# 
#     my_layout = layout_grid_to_dict(my_layout_grid)
# 
# Then sanity-check it: print `my_layout['agent_idx']` (must have length 2), and confirm each
# of `pot_idx`, `plate_pile_idx`, `goal_idx`, and whichever of `onion_pile_idx`/`tomato_pile_idx`
# you plan to use, is non-empty.
# 
# IMPLICATIONS: `my_layout` is now the *exact* object accepted by
# `Overcooked_v2(layout=..., ...)`. If you skip cell 6 (registration) you can already pass this
# dict directly to `jaxmarl.make('overcooked_v2', layout=my_layout, ...)` from Python. What
# registration buys you is the ability to reference it by name from the yaml config, which is
# how the training script consumes it.
# ------------------------------------------------------------------------------



## 5. Authoring path B — build the layout dict directly (no ASCII grid)

The parser is a convenience. Since you're building from scratch, you may want to skip it
entirely — e.g. because you want to generate layouts procedurally, or your design doesn't fit
the 'rectangular ASCII' assumption, or you want to place stations at positions the parser
won't naturally give you (e.g. multi-cell interactive zones, custom densities).

The dict you produce here must match the section-2 schema *exactly*. The example below
reconstructs the `cramped_room_v3` layout by hand so you can see the parser's output in raw
form.


In [ ]:
# EXISTING schema, written out by hand. This is byte-for-byte equivalent to
# `layout_grid_to_dict(cramped_room_v3)` — verified in the assert below.
#
# Grid (5x5, row-major, idx = row*5 + col):
#   row 0 (idx  0-4 ): W W W W W
#   row 1 (idx  5-9 ): P _ _ _ O          (5=P pot, 9=O onion)
#   row 2 (idx 10-14): W _ _ _ W
#   row 3 (idx 15-19): B A _ A W          (15=B plate, 16=A agent, 18=A agent)
#   row 4 (idx 20-24): W W X W W          (22=X goal)

manual_cramped_room_v3 = FrozenDict({
    'height': 5,
    'width':  5,
    # every wall AND every station cell goes in here
    'wall_idx': jnp.array(
        [0, 1, 2, 3, 4,          # top wall
         5, 9,                    # row-1 walls-that-are-stations (pot @5, onion @9)
         10, 14,                  # row-2 side walls
         15, 19,                  # row-3 walls-that-are-stations (plate @15) + right wall
         20, 21, 22, 23, 24],     # bottom wall (goal @22 is here too)
        dtype=jnp.int32),
    'agent_idx':       jnp.array([16, 18], dtype=jnp.int32),
    'goal_idx':        jnp.array([22],     dtype=jnp.int32),
    'plate_pile_idx':  jnp.array([15],     dtype=jnp.int32),
    'onion_pile_idx':  jnp.array([9],      dtype=jnp.int32),
    'tomato_pile_idx': jnp.array([]),   # empty ↦ float32 is OK, constructor tolerates it
    'pot_idx':         jnp.array([5],      dtype=jnp.int32),
})

# Prove it matches the parser output.
parsed = overcooked_v2_layouts['cramped_room_v3']
for k in ['height','width','agent_idx','goal_idx','plate_pile_idx','onion_pile_idx','pot_idx']:
    a, b = manual_cramped_room_v3[k], parsed[k]
    same = bool(jnp.all(jnp.asarray(a) == jnp.asarray(b))) if hasattr(a,'shape') else a == b
    print(f'  {k:>18}: manual == parsed ? {same}')


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# If you're going the from-scratch dict route, build `my_layout` here as a `FrozenDict` with
# the 9 schema keys. A useful helper — flat index arithmetic:
# 
#     def rc(r, c, width): return r * width + c
# 
#     H, W = 6, 7
#     my_layout = FrozenDict({
#         'height': H,
#         'width':  W,
#         'wall_idx': jnp.array([...], dtype=jnp.int32),   # perimeter + interior walls + station cells
#         'agent_idx': jnp.array([rc(3, 2, W), rc(3, 4, W)], dtype=jnp.int32),
#         'goal_idx': jnp.array([rc(5, 3, W)], dtype=jnp.int32),
#         'plate_pile_idx':  jnp.array([...], dtype=jnp.int32),
#         'onion_pile_idx':  jnp.array([...], dtype=jnp.int32),
#         'tomato_pile_idx': jnp.array([]),
#         'pot_idx':         jnp.array([...], dtype=jnp.int32),
#     })
# 
# If you already wrote `my_layout` via path A (cell after ASCII parse), you can skip this cell
# entirely — the two paths produce the same object and the trainer can't tell them apart.
# 
# IMPLICATIONS OF PATH B vs PATH A:
#   - Path A (parser) enforces the wall-perimeter and station-in-wall-idx invariants for you.
#     Path B doesn't — you have to remember to duplicate each station cell into `wall_idx` or
#     agents will happily walk onto pots/goals and the collision code will misbehave.
#   - Path B lets you place *multiple* stations of the same type at arbitrary positions
#     (e.g. 3 pots along a diagonal) which is awkward to draw in ASCII.
#   - Path B is the only route if you want to *generate* layouts programmatically — see
#     `generate_variants` in `layouts.py:593` for how the existing 2600 variants were made.
#   - Whichever path you pick, the schema conformance check above is your friend: run it once
#     on your custom dict to catch typos before the env constructor throws a cryptic KeyError.
# ------------------------------------------------------------------------------



## 6. Registering your layout so the trainer can find it by name

`baselines/IPPO/ippo_rnn_overcooked_v2.py:1059-1060` does:

```python
layout_name = config['ENV_KWARGS']['layout']            # e.g. 'cramped_room_v3'
config['ENV_KWARGS']['layout'] = overcooked_v2_layouts[layout_name]
```

i.e. the yaml value is a *string key* that must exist in the `overcooked_v2_layouts` dict.
If your layout isn't registered, you'll get a `KeyError` at line 1060.

The dict itself is built in `jaxmarl/environments/overcooked_v2/layouts.py:660`. You can either
(a) add a permanent entry there, or (b) mutate the imported dict from Python — both work
identically at runtime because it's the same object.


In [ ]:
# EXISTING pattern — how `cramped_room_v3` gets into the registry
# (extracted from layouts.py:663).
print('cramped_room_v3' in overcooked_v2_layouts,
      '<- already registered at import time')
print('registry size:', len(overcooked_v2_layouts))


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Register your layout under a name of your choice:
# 
#     overcooked_v2_layouts['my_layout'] = my_layout
# 
# IMPLICATIONS:
#   - This mutates the imported dict for the lifetime of the Python process. It is not
#     persistent — the trainer, run as a separate `python ...` invocation, will not see it
#     unless you also add the same line to `jaxmarl/environments/overcooked_v2/layouts.py`
#     (near line 660). For the notebook-only sanity checks below this in-memory registration is
#     enough.
#   - Once registered, you can set `ENV_KWARGS.layout: my_layout` in the yaml and everything
#     downstream just works.
#   - Choose a fresh name; overwriting an existing key (e.g. `cramped_room_v3`) silently
#     changes what every other script sees.
# ------------------------------------------------------------------------------



## 7. Recipes — what combination of ingredients pays out

Pots emit reward only when the ingredients inside them match a *recipe id* you've enabled.
The lookup table lives at `overcooked.py:278-289`:

| id | contents                    |
|----|-----------------------------|
| 11 | 1 onion                     |
| 12 | 2 onions                    |
| 13 | 3 onions                    |
| 14 | 1 tomato                    |
| 15 | 2 tomatoes                  |
| 16 | 3 tomatoes                  |
| 17 | 1 onion + 1 tomato          |
| 18 | 2 onions + 1 tomato         |
| 19 | 1 onion + 2 tomatoes        |

`recipes` in `ENV_KWARGS` is a list of these ids as *strings*, e.g. `['11']` in the current
yaml. In the constructor these ids flip the corresponding row of `lookup_reward` to `1`,
meaning that dish now grants reward on delivery.


In [ ]:
# EXISTING config (from ippo_rnn_overcooked_v2.yaml): recipe id 11 = single-onion soup.
current_recipes = ['11']
print('recipes in current training config:', current_recipes)
print("agent must:  pick onion -> put in pot -> wait for cook -> pick plate -> collect -> deliver at X")


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Pick recipes for your task:
# 
#     my_recipes = ['11']         # single-onion soup, matches current default
#     # my_recipes = ['11', '17'] # onion soup OR onion+tomato — multi-recipe task
# 
# IMPLICATIONS:
#   - Every recipe you enable must be *actually cookable* on your grid. Enabling '17'
#     (onion+tomato) but omitting `T` from the layout means the agent can never earn that
#     reward.
#   - Enabling multiple recipes turns the task into a choice/decision problem — the agent's
#     RNN has to figure out which dish is worth making. This is one of the axes the paper
#     manipulates when studying whether partner modelling emerges.
#   - `recipes` must be a list of strings (the env constructor does `int(r) for r in recipes`
#     at `overcooked.py:277`). Passing bare ints works from Python, but hydra loads strings
#     from yaml, so keep the string convention if you plan to configure via yaml.
# ------------------------------------------------------------------------------



## 8. Per-agent reward shaping (subtask specialisation)

The env exposes two 5-vectors that shape *dense* per-agent reward for the sub-events that lead
up to a delivery. Both are constructor kwargs of `Overcooked_v2`
(`overcooked.py:247-249`). The five slots correspond to:

1. put an **onion** in the pot
2. put a **tomato** in the pot
3. pick up a **plate**
4. pick up a **dish** (cooked soup) from the pot
5. (unused / reserved)

The paper's central manipulation lives in `behaviour_modes_agent_{0,1}` (`overcooked.py:316-324`).
Each mode row is a shaping vector; the training loop switches modes over time so that different
partner styles emerge (e.g. one agent specialises in onion→pot, the other in plating).
This is what the *partner modelling* part of the RNN has to pick up on.


In [ ]:
# EXISTING defaults from overcooked.py:247-249 and :316-324.
default_shaping_agent_0 = jnp.array([3, 3, 3, 5, 0])
default_shaping_agent_1 = jnp.array([1, 1, 1, 2, 0])

behaviour_modes_agent_0 = jnp.array([
    [-10, -10, -10,  1,  1],   # mode 0: PENALISE putting ingredients in the pot, reward plating/dishing
    [ 1,   1,   1, -10, -10],  # mode 1: reward ingredient handling, penalise plating/dishing
])
behaviour_modes_agent_1 = jnp.array([
    [-10, -10, -10,  1,  1],
    [ 1,   1,   1, -10, -10],
])
print('shaping vectors have shape:', default_shaping_agent_0.shape)
print('behaviour_modes shape (n_modes, 5):', behaviour_modes_agent_0.shape)


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Optional — override the shaping vectors when you build the env:
# 
#     my_shaping_0 = jnp.array([3, 3, 3, 5, 0])   # keep default
#     my_shaping_1 = jnp.array([1, 1, 1, 2, 0])
# 
# You can also edit the two `behaviour_modes_*` arrays inside `overcooked.py` if you want
# different specialisation splits (e.g. onion-only vs tomato-only).
# 
# IMPLICATIONS:
#   - Shaping reward is *annealed to zero* by the trainer over `REW_SHAPING_HORIZON` steps
#     (see `ippo_rnn_overcooked_v2.py:539-541`). Changing the values shifts *what the agents
#     learn during the warm-up*, but final policies are still evaluated on sparse recipe reward.
#   - If you change the *shape* of these vectors (still 5, but different semantics), you must
#     also update the reward emission inside `env.step` — this is the boundary where 'safe env
#     tweak' ends and 'you're changing the training signal' begins.
#   - The `behaviour_modes_*` matrices are what the paper's whole thesis rests on: they create
#     the *partner variability* that the RNN eventually has to model. If you flatten them
#     (identical rows), you remove the reason a partner model would emerge — a useful ablation.
# ------------------------------------------------------------------------------



## 9. Building the env — the trainer's contract

Reproducing the two lines the trainer uses (`ippo_rnn_overcooked_v2.py:502,512`):

```python
env = jaxmarl.make(config['ENV_NAME'], **config['ENV_KWARGS'])
env = LogWrapper(env, replace_info=False)
```

After this the trainer reads only two things off `env`: its **observation shape** and its
**action count**. Anything you do that changes those numbers will break weight loading.


In [ ]:
# EXISTING trainer path, executed inline so you can inspect the env object.
env = jaxmarl.make(
    'overcooked_v2',
    layout=overcooked_v2_layouts['cramped_room_v3'],  # ← use the REGISTERED dict, not the string
    recipes=['11'],
    max_steps=400,
    random_reset=True,
)
env = LogWrapper(env, replace_info=False)

print('observation_space().shape :', env.observation_space().shape,
      '   ← (W, H, 28); trainer inits the CNN off this')
print('action_space(agent_0).n   :', env.action_space(env.agents[0]).n,
      '   ← 6 discrete actions (up/down/right/left/stay/interact)')
print('num_agents                :', env.num_agents)
print('agents                    :', env.agents)


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Build YOUR env the same way:
# 
#     my_env = jaxmarl.make(
#         'overcooked_v2',
#         layout=overcooked_v2_layouts['my_layout'],
#         recipes=my_recipes,
#         max_steps=400,
#         random_reset=True,
#     )
#     my_env = LogWrapper(my_env, replace_info=False)
# 
# Then assert the contract:
# 
#     assert my_env.observation_space().shape[-1] == 28
#     assert my_env.action_space(my_env.agents[0]).n == 6
# 
# IMPLICATIONS:
#   - `random_reset=True` samples fresh agent positions every episode; `False` uses the exact
#     `agent_idx` from your layout. The paper uses `True` — turning it off makes the task
#     much easier and can wash out partner modelling effects.
#   - `max_steps=400` is what the paper trains on; changing it changes the per-episode return
#     scale, which interacts with `GAMMA=0.99` and the shaping anneal.
#   - `LogWrapper(replace_info=False)` preserves `info` dict entries the trainer reads for
#     metrics logging. Do NOT flip it to True unless you also strip the metric-logging code
#     downstream.
# ------------------------------------------------------------------------------



## 10. Sanity rollout — verify a random policy can ever score

Before you spend an hour on GPU training, run a purely random policy on your env and confirm
that (a) it doesn't crash, (b) episodes terminate at `max_steps`, and (c) at least occasionally
you see `reward > 0`. If you never see a positive reward, no amount of RL will help — the
task is unsolvable as configured.


In [ ]:
# EXISTING-style rollout — mirrors what the trainer's inner scan does, just with random actions.
key = jax.random.PRNGKey(0)
key, key_r = jax.random.split(key)
obs, state = env.reset(key_r)

total_reward = 0.0
any_positive = False
for t in range(200):
    key, key_a0, key_a1, key_s = jax.random.split(key, 4)
    actions = {
        'agent_0': jax.random.randint(key_a0, (), 0, env.action_space(env.agents[0]).n),
        'agent_1': jax.random.randint(key_a1, (), 0, env.action_space(env.agents[1]).n),
    }
    obs, state, reward, done, info = env.step(key_s, state, actions)
    r0 = float(reward['agent_0']); r1 = float(reward['agent_1'])
    total_reward += r0 + r1
    if r0 > 0 or r1 > 0:
        any_positive = True
        print(f't={t}  r0={r0:+.2f}  r1={r1:+.2f}')
print(f'total_reward over 200 steps: {total_reward:+.2f}   any_positive={any_positive}')


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Repeat the rollout with YOUR env. Bump the loop to ~2000 steps so you have a decent chance
# of a random policy accidentally completing the recipe.
# 
# IMPLICATIONS:
#   - `any_positive == False` after several thousand random steps almost always means the
#     layout is broken: pot not reachable, plate pile inaccessible, goal walled off, or the
#     recipe requires an ingredient you didn't place. Fix the layout HERE, before training —
#     the trainer will happily burn GPU hours on an unsolvable task and give you flat curves.
#   - The dense shaping reward WILL make individual r > 0 events much more common than sparse
#     delivery reward, so seeing r=+1 mostly proves the shaping paths are working, not that
#     the full recipe pipeline pays out. Look for a large reward event (>= 5) at least once.
# ------------------------------------------------------------------------------



## 11. (Optional) Render a frame

`OvercookedVisualizer` from `jaxmarl/viz/overcooked_visualizer_v2.py` renders a state to a
numpy image. Handy for debugging where your agents actually spawn.


In [ ]:
# EXISTING visualiser class. Render the current post-rollout state.
viz = OvercookedVisualizer()
try:
    frame = viz.render(agent_view_size=5, state=state.env_state if hasattr(state, 'env_state') else state, highlight=False)
    print('rendered frame shape:', getattr(frame, 'shape', type(frame)))
except Exception as e:
    # Rendering can fail on some layouts / on CPU-only nodes without display libs.
    print('rendering skipped:', type(e).__name__, e)


In [ ]:
# ------------------------------------------------------------------------------
# YOUR TURN
# Optional — render YOUR env at reset and after N random steps to eyeball the geometry.
# 
#     obs, s = my_env.reset(jax.random.PRNGKey(0))
#     frame = viz.render(agent_view_size=5, state=s.env_state, highlight=False)
#     import matplotlib.pyplot as plt; plt.imshow(frame); plt.axis('off'); plt.show()
# 
# IMPLICATIONS: rendering is purely for you — nothing in the trainer depends on it. Feel free
# to skip if you're only running headless on the cluster.
# ------------------------------------------------------------------------------



## 12. Plugging your env into the training pipeline

Once cells 2–8 all pass, wiring the env into training is *three* edits:

### (a) Persist the layout registration

Add your grid + registration to `jaxmarl/environments/overcooked_v2/layouts.py` alongside
`cramped_room_v3` (line ~156) and inside the `overcooked_v2_layouts` dict (line ~660):

```python
my_layout_grid = """
...your grid...
"""

overcooked_v2_layouts = {
    ...
    'my_layout': layout_grid_to_dict(my_layout_grid),
}
```

The mutation you did in cell 4 does NOT survive across process boundaries — the trainer runs
in a fresh Python process, so the registration must live on disk.

### (b) Point the yaml at it

In `baselines/IPPO/config/ippo_rnn_overcooked_v2.yaml`:

```yaml
ENV_KWARGS:
  layout: 'my_layout'      # <- name you just registered
  recipes: ['11']          # <- your chosen recipe ids
  max_steps: 400
  random_reset: True
```

Also disable checkpoint loading unless you're intentionally warm-starting from a compatible
run — set `LOAD_MODEL: False` and `RNN_LOAD_MODEL: False`, because those checkpoints were
trained on `cramped_room_v3` and their weights are only shape-compatible if your grid has
the same width/height.

### (c) Run

```bash
cd baselines/IPPO
python ippo_rnn_overcooked_v2.py
```

Nothing below `env = jaxmarl.make(...)` in that script should need to change. The RNN,
PPO update, logging, and checkpoint-save code all consume `env` through the same three
properties you already verified in cell 7 (`observation_space`, `action_space`, `num_agents`).

If you want to change reward *dynamics* (not just the shaping vectors, but e.g. when reward
is emitted, or a completely new subtask), that's a `overcooked.py` edit — that file is the
actual boundary between 'plug in a new task' and 'change the RL problem'.


## 13. Pre-flight checklist

Before you kick off training, verify each of these — the training script won't tell you
any of them are wrong; it will just train badly.

- [ ] (Path A) `my_layout_grid` has exactly 2 `A`, rows all the same width, wall perimeter,
      OR (Path B) `my_layout` is a `FrozenDict` with exactly the 9 keys from the section-2 schema.
- [ ] Every station cell appears in `wall_idx` (path A does this for you; path B is your job).
- [ ] `agent_idx` has exactly 2 entries.
- [ ] `my_layout` has non-empty index arrays for each station you actually need for your recipes.
- [ ] Every enabled recipe id can be cooked from the ingredients present in the grid.
- [ ] `env.observation_space().shape[-1] == 28` and `env.action_space(agent_0).n == 6`.
- [ ] Random-policy rollout produces at least one reward event > 0 in a few thousand steps.
- [ ] Layout registration written into `layouts.py`, not just done in-notebook.
- [ ] `ENV_KWARGS` in the yaml points at your registered name, not a stale one.
- [ ] `LOAD_MODEL` / `RNN_LOAD_MODEL` set appropriately for your grid size.

When all boxes are ticked, the RNN training loop is guaranteed to *see* your env; whether
it *learns* on it is now a research question, not a plumbing one.
